# Structured Output with Amazon Bedrock Custom Model Import

This notebook shows how to use [Amazon Bedrock structured output](https://docs.aws.amazon.com/bedrock/latest/userguide/custom-model-import-advanced-features.html#structured-outputs) to constrain a model's output (both foundation and fine-tuned) to a JSON schema. The schema is enforced at the token level during decoding, so every response is guaranteed to parse and conform to your schema — no regex extraction, no malformed-output retries.


**Use case:** extracting structured metadata (entities, topics, sentiment, etc.) from a news article. The same pattern applies to any extraction task — invoices, support tickets, resumes, research abstracts.

**Pipeline:**
```
article text → CMI model + json_schema constraint → validated JSON
```

**Model & API format:** this notebook runs against `qwen.qwen3-coder-30b-a3b-v1:0` for easy demonstration, called via `InvokeModel` using the `OpenAIChatCompletion` request/response format (`{messages, max_tokens, response_format}`). 

CMI also supports `OpenAICompletion` and `BedrockCompletion` formats — see [Invoke an imported model](https://docs.aws.amazon.com/bedrock/latest/userguide/invoke-imported-model.html) for the differences and [Structured outputs for imported models](https://docs.aws.amazon.com/bedrock/latest/userguide/custom-model-import-advanced-features.html#structured-outputs) for how `response_format` varies between them. 

To use your own fine-tuned model, replace `MODEL_ARN` with your CMI model ARN — as long as it accepts the `OpenAIChatCompletion` shape, no other changes are required.

**References:**
1. [Invoke an imported model (request/response formats)](https://docs.aws.amazon.com/bedrock/latest/userguide/invoke-imported-model.html)
2. [Structured outputs for Bedrock imported models](https://docs.aws.amazon.com/bedrock/latest/userguide/custom-model-import-advanced-features.html#structured-outputs)
3. [Get validated JSON results from models](https://docs.aws.amazon.com/bedrock/latest/userguide/structured-output.html)
4. [Structured outputs on Amazon Bedrock: Schema-compliant AI responses](https://aws.amazon.com/blogs/machine-learning/structured-outputs-on-amazon-bedrock-schema-compliant-ai-responses/)

## 1. Setup

In [1]:
%pip install -q boto3 pydantic

Note: you may need to restart the kernel to use updated packages.


In [12]:
import json
import time
from typing import Literal

import boto3
from botocore.config import Config as BotoConfig
from pydantic import BaseModel, Field

# !!!!!!!! REPLACE WITH YOUR CMI MODEL ARN !!!!!!!!
# i.e.: MODEL_ARN = "arn:aws:bedrock:us-west-2:<account>:imported-model/<id>"

# For demo purposes we use a foundation model so you can run the notebook — the API shape is identical.
MODEL_ARN = "qwen.qwen3-coder-30b-a3b-v1:0" # "<YOUR MODEL ARN>" 
REGION = "us-east-1"

session = boto3.Session()
client = session.client(
    "bedrock-runtime",
    region_name=REGION,
    config=BotoConfig(read_timeout=300, retries={"max_attempts": 0}),
)

In [13]:
def invoke(payload, max_retries=6):
    """Call invoke_model with retry on transient errors.

    Imported CMI models may cold-start; ModelNotReady errors are retried with a longer backoff.
    """
    last = None
    for attempt in range(max_retries):
        try:
            resp = client.invoke_model(
                modelId=MODEL_ARN,
                body=json.dumps(payload),
                contentType="application/json",
                accept="application/json",
            )
            return json.loads(resp["body"].read())
        except Exception as e:
            last = e
            msg = str(e)
            wait = 30 if "ModelNotReady" in msg else 10
            print(f"  attempt {attempt+1}: {msg[:100]}... retrying in {wait}s")
            time.sleep(wait)
    raise RuntimeError(f"failed after {max_retries} attempts: {last}")


def extract_content(result):
    """Return assistant text from an OpenAIChatCompletion response.

    Reasoning/CoT models may emit output in `reasoning` or `reasoning_content`
    instead of (or in addition to) `content`. Fall back to those if `content` is empty.
    """
    msg = result["choices"][0]["message"]
    text = msg.get("content") or ""
    if not text:
        reasoning = msg.get("reasoning") or msg.get("reasoning_content") or ""
        if reasoning and "{" in reasoning:
            text = reasoning
    return text

## 2. Example input

A short fictional news article. In a real pipeline this could come from an RSS feed, a CMS, or a scraped page.

In [14]:
ARTICLE = """Acme Robotics announced today that it has raised $120 million in a Series C funding round
led by Northwind Ventures, with participation from existing investors Pinegrove Capital and Helios Partners.
The Boston-based company, founded in 2018 by CEO Maria Chen and CTO David Okonkwo, plans to use the funding
to expand its warehouse automation business into the European market and double its engineering team by
the end of 2026.

"This round gives us the runway to bring our autonomous picking systems to retailers across the EU,"
said Chen in a statement. "Demand from logistics operators has tripled in the last 18 months."

The deal, which closed on May 12, 2026, values Acme at approximately $850 million. The company
previously raised $45 million in a Series B led by Pinegrove in 2023."""

SYSTEM_PROMPT = """You are an information extraction system. You read news articles and extract
structured metadata: headline, summary, entities (people, organizations, locations), topics,
sentiment, and any monetary or date values mentioned.

Respond with a single JSON object only. Do not include prose, markdown fences, or commentary."""

USER_PROMPT = f"""Extract structured metadata from the following article and return it as JSON:

{ARTICLE}"""

## 3. Baseline — prompted JSON (no schema constraint)

The prompt asks for JSON, but nothing forces it. Models often comply *most* of the time — and then drift into prose, markdown fences, extra commentary, or subtly malformed JSON whenever the input is unfamiliar or the temperature is high. Downstream code typically has to regex-extract a JSON block and hope it parses.

In [15]:
baseline_payload = {
    "messages": [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_PROMPT},
    ],
    "max_tokens": 4096,
    "temperature": 0.7,
}

t0 = time.time()
baseline_result = invoke(baseline_payload)
print(f"Baseline done in {time.time()-t0:.1f}s\n")
print(extract_content(baseline_result))

Baseline done in 3.4s

{
  "headline": "Acme Robotics Raises $120 Million in Series C Funding",
  "summary": "Acme Robotics secured $120 million in Series C funding led by Northwind Ventures, with participation from Pinegrove Capital and Helios Partners. The Boston-based company plans to expand into the European market and double its engineering team by 2026.",
  "entities": {
    "people": ["Maria Chen", "David Okonkwo"],
    "organizations": ["Acme Robotics", "Northwind Ventures", "Pinegrove Capital", "Helios Partners"],
    "locations": ["Boston", "EU"]
  },
  "topics": ["funding", "robotics", "warehouse automation", "expansion"],
  "sentiment": "positive",
  "monetary_values": [120000000, 850000000, 45000000],
  "date_values": ["2026-05-12", "2023", "2026"]
}


## 4. Define the output as a Pydantic schema

Constrained decoding requires a JSON schema. Pydantic generates one automatically via `model_json_schema()`, which gives us a single source of truth for both the constraint and downstream validation.

In [16]:
class MonetaryValue(BaseModel):
    amount: float = Field(description="Numeric amount")
    currency: str = Field(description="ISO 4217 currency code, e.g. USD, EUR")
    context: str = Field(description="What the amount refers to, e.g. 'Series C funding'")


class Person(BaseModel):
    name: str
    role: str | None = Field(default=None, description="Title or role if mentioned, e.g. 'CEO'")
    organization: str | None = Field(default=None, description="Affiliated organization if mentioned")


class Entities(BaseModel):
    people: list[Person]
    organizations: list[str]
    locations: list[str]


class Article(BaseModel):
    headline: str = Field(max_length=200, description="Concise headline summarizing the article")
    summary: str = Field(description="Two-to-three sentence summary of the article")
    category: Literal[
        "business", "technology", "politics", "sports", "science", "health", "entertainment", "other"
    ]
    sentiment: Literal["positive", "negative", "neutral", "mixed"]
    topics: list[str] = Field(description="Short topic keywords, e.g. 'funding', 'robotics'")
    entities: Entities
    monetary_values: list[MonetaryValue]
    dates_mentioned: list[str] = Field(description="Dates referenced in the article, ISO 8601 if possible")


article_schema = {"name": "Article", "schema": Article.model_json_schema()}
print(json.dumps(article_schema, indent=2)[:1200], "...")

{
  "name": "Article",
  "schema": {
    "$defs": {
      "Entities": {
        "properties": {
          "people": {
            "items": {
              "$ref": "#/$defs/Person"
            },
            "title": "People",
            "type": "array"
          },
          "organizations": {
            "items": {
              "type": "string"
            },
            "title": "Organizations",
            "type": "array"
          },
          "locations": {
            "items": {
              "type": "string"
            },
            "title": "Locations",
            "type": "array"
          }
        },
        "required": [
          "people",
          "organizations",
          "locations"
        ],
        "title": "Entities",
        "type": "object"
      },
      "MonetaryValue": {
        "properties": {
          "amount": {
            "description": "Numeric amount",
            "title": "Amount",
            "type": "number"
          },
          "currency": {

## 5. Generate with structured output

Pass the schema via `response_format`. The model is now constrained to emit valid JSON matching the schema.

In [17]:
structured_payload = {
    "messages": [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_PROMPT},
    ],
    "max_tokens": 4096,
    "temperature": 0.3,
    "response_format": {
        "type": "json_schema",
        "json_schema": article_schema,
    },
}

t0 = time.time()
structured_result = invoke(structured_payload)
print(f"Structured output done in {time.time()-t0:.1f}s\n")

structured_json = extract_content(structured_result)
print(structured_json)

Structured output done in 2.2s

{
  "headline": "Acme Robotics Raises $120 Million in Series C Funding",
  "summary": "Acme Robotics secured $120 million in Series C funding led by Northwind Ventures, with participation from Pinegrove Capital and Helios Partners. The Boston-based company plans to expand into the European market and double its engineering team by 2026.",
  "category": "technology",
  "sentiment": "positive",
  "topics": ["funding", "robotics", "expansion", "venture capital"],
  "entities": {
    "people": [
      {"name": "Maria Chen", "role": "CEO"},
      {"name": "David Okonkwo", "role": "CTO"}
    ],
    "organizations": [
      "Acme Robotics",
      "Northwind Ventures",
      "Pinegrove Capital",
      "Helios Partners"
    ],
    "locations": [
      "Boston",
      "EU"
    ]
  },
  "monetary_values": [
    {"amount": 120000000, "currency": "USD", "context": "Series C funding"},
    {"amount": 850000000, "currency": "USD", "context": "company valuation"},
    {

## 6. Validation

Sanity checks: the output is valid JSON and matches the Pydantic schema.

In [18]:
parsed = json.loads(structured_json)
print("JSON parse:        OK")

article = Article.model_validate(parsed)
print("Schema validation: OK")

print(f"\nHeadline:  {article.headline}")
print(f"Category:  {article.category}")
print(f"Sentiment: {article.sentiment}")
print(f"People:    {[p.name for p in article.entities.people]}")
print(f"Orgs:      {article.entities.organizations}")

JSON parse:        OK
Schema validation: OK

Headline:  Acme Robotics Raises $120 Million in Series C Funding
Category:  technology
Sentiment: positive
People:    ['Maria Chen', 'David Okonkwo']
Orgs:      ['Acme Robotics', 'Northwind Ventures', 'Pinegrove Capital', 'Helios Partners']


## Automatic CMI performance improvements

Per the [CMI performance enhancements announcement](https://aws.amazon.com/blogs/machine-learning/enhanced-performance-for-amazon-bedrock-custom-model-import/) (Nov 2025), existing imported models automatically benefit from PyTorch compilation + CUDA graph optimizations with no re-import required. Reported gains on reference models: TTFT reduced 76–88%, end-to-end latency reduced 18–59%, throughput up 25–29%. There is a one-time longer cold start on first deploy while artifacts compile and cache, then every scale-up after is faster than before.